In [ ]:
# --- 1. CONFIG AND DEFINITIONS ---

# 1.1 Dataset and data root config
import random
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import time

dataset = 'monk1'
data_root = './data'
use_optuna = False
use_grid = True
use_random = False
use_pca = False
monk3_reg = True
seed = random.randint(0, 100000) # Used via parameter injection script for generating models. The seed is saved in a json file for reproducibility. 


In [ ]:
# 1.2 Config limits in case of monk3 regularization
if dataset == 'monk3' and monk3_reg:
    C_LIMIT = 1.0
    GAMMA_LIMIT = 0.5 
else:
    C_LIMIT = 100.0
    GAMMA_LIMIT = 1.0


In [ ]:
# 1.3 Mean Euclidean Error Definition
import numpy as np

def mean_euclidean_error(y_true, y_pred):
    # Calculate the difference between true and predicted values
    diff = y_true - y_pred
    
    # Reshapes row vectors to column vectors for consistent norm calculation
    if diff.ndim == 1:
        diff = diff.reshape(-1, 1)

    # Calculate mean Euclidean error
    return np.linalg.norm(diff, axis=1).mean()

In [ ]:
# 1.4 Scorer for cross-validation
from sklearn.metrics import make_scorer

mee_scorer = make_scorer(mean_euclidean_error, greater_is_better=False)

In [ ]:
# === 2 IMPORTS AND UTILITIES ===
from utils.data_loader import get_monk_data, get_ml_cup_data  
from torch.utils.data import DataLoader, TensorDataset
from models import SVCModel, SVRModel
from sklearn.metrics import *
from sklearn.multioutput import MultiOutputRegressor
  
    
# 2.1 Utility function to extract data from DataLoader to NumPy arrays
def extract_data_to_numpy(data_loader):
    """
    Converts data from a PyTorch DataLoader into a flattened NumPy array pair (X, y).
    Targets (y) are returned in their original dimensionality (e.g., [N, M] for multi-output).
    """
    X_list = []
    y_list = []
    for X, y in data_loader:
        # Flatten the input (e.g., 28x28 image -> 784 features)
        X_list.append(X.view(X.size(0), -1).numpy()) 
        # Convert labels to NumPy
        y_list.append(y.numpy())
    
    X_data = np.concatenate(X_list)
    y_data = np.concatenate(y_list)
    
    # We return y_data as is (2D array, e.g., [N, 1] or [N, M]).
    return X_data, y_data

In [ ]:
# 2.2 StandardScaler import and initialization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
# 2.3 Data Loading and Preparation

from utils.data_loader import get_monk_data, get_ml_cup_data
import sys

dataset_name = dataset
BATCH_SIZE = 1024 # Batch size for DataLoader (not used in SVM but for consistency) 

print(f"Loading dataset: {dataset_name.upper()}...")
    
# Determine task type and load data (DataLoader objects are returned)
if dataset_name == 'monk1':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk_data(1, BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"

elif dataset_name == 'monk2':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk_data(2, BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"

elif dataset_name == 'monk3':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk_data(3, BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"

elif dataset_name == 'mlc25':
    train_loader, _, test_loader, INPUT_SIZE, OUTPUT_SIZE, _ = get_ml_cup_data(BATCH_SIZE, data_root, validation_ratio=0.0, test_ratio=0.25, scaler=scaler, scale_target=False)
    is_regression_task = True
    metric_name = "Test MEE"

else:
    # This block handles the error if the dataset is outside the specified choices (monk1, monk2, monk3, mlc25).
    raise ValueError("Unsupported dataset for SVM.")


# 2.4 Data Extraction to NumPy Arrays

# Convert DataLoaders (PyTorch) to NumPy arrays (Scikit-learn)
X_train, y_train = extract_data_to_numpy(train_loader)
X_test, y_test = extract_data_to_numpy(test_loader)

print(f"Data loaded: Training samples={X_train.shape[0]}, Test samples={X_test.shape[0]}")

In [ ]:
# 2.5 PCA Import and Application
from sklearn.decomposition import PCA
import joblib

# Apply PCA if specified
pca_model = None

if use_pca:
    print("Applying PCA for dimensionality reduction...")
    pca_model = PCA(n_components=2, random_state=seed)  # Retain 2 principal components as suggested 
    print(f"Original feature dimensions: {X_train.shape[1]}")
    
    # Transform the data
    X_train = pca_model.fit_transform(X_train)
    X_test = pca_model.transform(X_test)
    
    print(f"PCA applied: Reduced feature dimensions to {X_train.shape[1]}")
   
else:
    print("PCA not applied.")

In [ ]:
# 2.6 Verify shapes

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
# 2.7 Directory setup for saving results and models
import os   
import datetime

base_dir = "./svm_saved_models"
task_subfolder = "regression" if is_regression_task else "classification"
preprocess_subfolder = "PCA" if use_pca else "Standard"
method_subfolder = "Optuna" if use_optuna else "RandomSearch" if use_random else "GridSearch"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_type = "SVR" if is_regression_task else "SVC"

reg_suffix = ""
if dataset == 'monk3' and monk3_reg:
    reg_suffix = "_REG"

run_name = f"{model_type}_{reg_suffix}_{timestamp}"
if 'monk' in dataset:
    run_dir = os.path.join(base_dir, task_subfolder, preprocess_subfolder, method_subfolder, dataset.upper(), run_name)
else: 
    run_dir = os.path.join(base_dir, task_subfolder, preprocess_subfolder, method_subfolder, run_name)
os.makedirs(run_dir, exist_ok=True)


In [ ]:
# 2.8 Kfold setup for cross-validation
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) if not is_regression_task else KFold(n_splits=5, shuffle=True, random_state=seed)     

In [ ]:
# 3. GRID SEARCH (BASELINE - COARSE)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVC, SVR
import numpy as np


if use_grid:
    print(f"\n --- Grid Search Execution (Baseline) for {dataset} ---")
    start_time_grid = time.time()

    # 3.1 Discrete value definitions
    if dataset == 'monk3' and monk3_reg:
        C_grid = np.logspace(-3, 0, 4)  # from 0.001 to 1.0
    else: 
        C_grid = np.logspace(-3, 2, 6) # from 0.001 to 100.0
    
    Gamma_float = np.logspace(-3, 0, 4) # from 0.001 to 1.0
    Epsilon_grid = np.logspace(-2, 1, 4) # from 0.01 to 10.0
    
    # 3.2 Common grid parameters for both SVC and SVR
    common_grid_params = [
        # 1. Linear
        {
            'kernel': ['linear'],
            'C': C_grid
        },
        # 2.A RBF Discrete Gamma
        {
            'kernel': ['rbf'],
            'C': C_grid,
            'gamma': ['scale', 'auto']
        },
        # 2.B RBF Continuous Gamma 
        {
            'kernel': ['rbf'],
            'C': C_grid,
            'gamma': Gamma_float
        },
        # 3.A Poly Discrete Gamma
        {
            'kernel': ['poly'],
            'C': C_grid,
            'gamma': ['scale', 'auto'],
            'degree': [2, 3],       
            'coef0': [0.0, 1.0, 5.0, 10.0]     
        },
        # 3.B Poly Continuous Gamma 
        {
            'kernel': ['poly'],
            'C': C_grid,
            'gamma': Gamma_float,
            'degree': [2, 3, 4, 5],
            'coef0': [0.0, 1.0, 5.0, 10.0]
        }
    ]

    if is_regression_task:
        grid_params = []
        for p in common_grid_params:
                p_copy = p.copy()
                p_copy['epsilon'] = Epsilon_grid 
                new_p = {f'estimator__{k}': v for k, v in p_copy.items()}
                grid_params.append(new_p)
        print("SVR parameters with epsilon added.")
    else:
        grid_params = common_grid_params
        print("SVC parameters without epsilon.")



In [ ]:
if use_grid:
    # 3.3 Base estimator definition
    limit = 100000
    size = 1000

    base_estimator = MultiOutputRegressor(SVR(max_iter=limit, cache_size=size)) if is_regression_task \
                else SVC(max_iter=limit, cache_size=size, probability = True, random_state = seed)

    # 3.4 GridSearchCV configuration
    grid_search = GridSearchCV(
        estimator = base_estimator,
        param_grid = grid_params, 
        cv = kfold,
        scoring = 'accuracy' if not is_regression_task else mee_scorer,
        n_jobs = -1,
        verbose = 1
    )


In [ ]:
if use_grid:
    # 3.5. Fit
    grid_search.fit(X_train, y_train)

    end_time_grid = time.time()
    time_grid = end_time_grid - start_time_grid

    print(f"Grid Search completed in {time_grid:.2f}s")
    


In [ ]:
# 3.6 Extract and clean best parameters
if use_grid:
    best_params_raw = grid_search.best_params_
    best_params_clean = {}

    if is_regression_task:
        # ML-CUP: extract the actual parameter names removing 'estimator__' prefix 
        for k, v in best_params_raw.items():
            clean_key = k.replace('estimator__', '')
            best_params_clean[clean_key] = v
    else:
        # MONK: no need to clean parameter names 
        best_params_clean = best_params_raw

    print(f"   Best Params (Clean): {best_params_clean}")

In [ ]:
# 3.7 Force wrap in custom classes
if use_grid:
    print("Wrapping results in custom classes (SVMModel/SVRModel)...")
    
    final_models_list = []
    
    if is_regression_task:
        num_targets = y_train.shape[1]
        is_multi_output = True
        
        for i in range(num_targets):
            print(f"Re-fitting Custom SVRModel for target {i}...")
            
            # 1. Set up custom wrapper with cleaned parameters
            custom_model = SVRModel(**best_params_clean)
            
            # 2. Select the specific target column
            if hasattr(y_train, 'iloc'):
                y_target = y_train.iloc[:, i]
            else:
                y_target = y_train[:, i]
            
            # 3. Fit
            custom_model.fit(X_train, y_target)
            
            # 4. Add to the list
            final_models_list.append(custom_model)
            
        # The best_model "global" for any generic reference in future cells
        best_model = final_models_list  
        best_cv_scores = [abs(grid_search.best_score_)] * len(final_models_list)
        
    else:
        print(f"Re-fitting Custom SVMModel...")
        is_multi_output = False
        
        # 1. Set up custom wrapper
        params_monk = best_params_clean.copy()
        if 'probability' not in params_monk:
             params_monk['probability'] = True
             
        custom_model = SVCModel(**params_monk, random_state=seed)
        
        # 2. Fit
        custom_model.fit(X_train, y_train)
        
        # 3. List setup with single element
        final_models_list = [custom_model]
        best_model = custom_model
        best_cv_scores = [abs(grid_search.best_score_)]
        
    # Support variables for the rest of the notebook
    hp_search = grid_search 
    best_params = best_params_clean
    

In [ ]:
# --- 4. RANDOM SEARCH---

# 4.1 Parameter Definition
if use_random:
    from sklearn.svm import SVC, SVR
    from scipy.stats import loguniform, uniform

    start_time_random = time.time()
    
    # 1. Definition of parameters in common for both SVC and SVR
    common_params = [
        # 1. Linear Kernel: C is the ONLY parameter. Gamma is implicitly 'scale' or ignored.
        {
            'kernel': ['linear'],
            'C': loguniform(1e-3, C_LIMIT),
        },
        
        # 2.A RBF Kernels with Discrete Gamma
        {
            'kernel': ['rbf'],
            'C': loguniform(1e-3, C_LIMIT),
            'gamma': ['scale', 'auto'], # Discrete strings only
        },
        # 2.B RBF Kernels with Continuous Gamma
        {
            'kernel': ['rbf'],
            'C': loguniform(1e-3, C_LIMIT),
            'gamma': loguniform(1e-3, GAMMA_LIMIT), # Continuous distribution object only
        },
        # 3.A Poly Kernels with Discrete Gamma
        {
            'kernel': ['poly'],
            'C': loguniform(1e-3, C_LIMIT),
            'gamma': ['scale', 'auto'], # Discrete strings only
            'degree': [2,3,4,5],
            'coef0': uniform(0, 10)
        },
        # 3.B Poly Kernels with Continuous Gamma  
        {
            'kernel': ['poly'],
            'C': loguniform(1e-3, C_LIMIT),
            'gamma': loguniform(1e-3, GAMMA_LIMIT), # Continuous distribution object only
            'degree': [2,3,4,5],
            'coef0': uniform(0, 10)
        }
    ]

    # 2. Conditional logic for epsilon parameter in SVR
    if is_regression_task:
        params = []
        for p in common_params:
            p_new = p.copy()
            p_new['epsilon'] = loguniform(1e-2,1e1)
            params.append(p_new)  # Append modified copy for SVR
        
        print("SVR parameters with epsilon added.")

    else:
        params = common_params
        print("SVC parameters without epsilon.")


    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) if not is_regression_task else KFold(n_splits=5, shuffle=True, random_state=seed)  

In [ ]:
# 4.2 Randomized Search configuration
if use_random:
    # HP Randomized Search Configuration

    limit = 100000
    size = 1000

    # Max iter and cache size set for efficiency
    base_estimator = SVC(max_iter=limit, cache_size=size, probability=True, random_state=seed) if not is_regression_task \
                else SVR(max_iter=limit, cache_size=size)

    hp_search = RandomizedSearchCV(
        estimator = base_estimator,
        param_distributions = params,
        n_iter = 200,                     # Number of random configurations to try
        scoring = 'accuracy' if not is_regression_task else mee_scorer,
        cv = kfold, 
        n_jobs = -1,
        verbose = 1,
        random_state = seed
     )                          


In [ ]:
# 4.3 Model Wrapping (if needed)
if use_random:
    if is_regression_task:
        # Wrap in MultiOutputRegressor if the dataset is mlc25
        final_model = MultiOutputRegressor(hp_search)  
    else:
        final_model = hp_search

In [ ]:
# 4.4 Execute Randomized Search and Fit Final Model
if use_random:
    final_model.fit(X_train, y_train)

    end_time_random = time.time()
    time_random = end_time_random - start_time_random
    print(f"Randomized Search completed in {time_random:.2f}s")
    best_cv_scores = []

    if is_regression_task:
        for estimator in final_model.estimators_:
            best_cv_scores.append(abs(estimator.best_score_))
    else:
        best_cv_scores.append(final_model.best_score_)

In [ ]:
# 4.5 Comprehensive Analysis and Saving of Random Search Results

if use_random:
    print("\n Saving Random Search Results & Plots...")
    
    # CSV and Plotting Helper Function
    def save_rs_analysis(cv_results, filename_prefix, title_suffix):
        # 1. Convert to DataFrame
        df_res = pd.DataFrame(cv_results)
        
        # 2. Save CSV
        csv_path = os.path.join(run_dir, f"{filename_prefix}.csv")
        df_res.to_csv(csv_path, index=False)
        print(f"   -> CSV saved: {csv_path}")
        
        # 3. Plotting
        params_of_interest = ['C', 'gamma', 'epsilon', 'degree', 'coef0']
        
        for param_name in params_of_interest:
            col_name = f"param_{param_name}"
            
            # Skip if parameter not in results
            if col_name not in df_res.columns or df_res[col_name].isna().all():
                continue
            
            # Prepare DataFrame for plotting
            df_plot = df_res.copy()
            df_plot[col_name] = pd.to_numeric(df_plot[col_name], errors='coerce')
            
            # Skip if all values are NaN after conversion
            if df_plot[col_name].isna().all():
                continue

            plt.figure(figsize=(10, 6))
            try:
                # Scatterplot: Parameter vs Score
                sns.scatterplot(
                    data=df_plot, 
                    x=col_name, 
                    y='mean_test_score', 
                    hue='param_kernel', 
                    style='param_kernel',
                    s=80, alpha=0.8, palette='viridis'
                )
                
                plt.title(f"Impact of {param_name}: {title_suffix}")
                plt.ylabel("CV Score")
                plt.grid(True, which="both", ls="-", alpha=0.2)
                
                # Log scale for certain parameters
                if param_name in ['C', 'gamma', 'epsilon']:
                    plt.xscale('log')
                    plt.xlabel(f"{param_name} parameter (log scale)")
                else:
                    plt.xlabel(f"{param_name} parameter (linear scale)")
                
                # Save Plot
                plot_filename = f"{filename_prefix}_scatter_{param_name}.png"
                plot_path = os.path.join(run_dir, plot_filename)
                plt.savefig(plot_path)
                plt.show()
                plt.close()
                
            except Exception as e:
                print(f"Plotting failed for {param_name}: {e}")
        
        print(f"   -> All plots saved in {run_dir}")

    # --- EXECUTION ---
    if is_regression_task:
        # Multi-Output: final_model contains N estimators
        for i, estimator in enumerate(final_model.estimators_):
            print(f"Processing Target {i}...")
            save_rs_analysis(
                estimator.cv_results_, 
                filename_prefix=f"rs_results_target_{i}",
                title_suffix=f"Target {i} (ML-CUP)"
            )
    else:
        # Single-Output: final_model is directly the RandomizedSearchCV
        print("Processing Single Target...")
        save_rs_analysis(
            final_model.cv_results_, 
            filename_prefix="rs_results_monk",
            title_suffix="MONK-1 Classification"
        )

In [ ]:
# 4.6 Print Best Parameters / Estimators
if use_random:
    if is_regression_task:
        # Print estimators for each output if mlc25
        print(final_model.estimators_)
    else:
        print(final_model.best_params_)
        

In [ ]:
# 4.7 Final Model Reconstruction from Best Estimator
if use_random:
    # --- 4.7.1 Helper Function to Create Wrapper from Estimator ---
    def create_model_from_estimator(best_estimator):
        """
        Extracts parameters from the best_estimator and creates an instance
        of the following custom wrapper (SVCModel or SVRModel).
        """
        # 1. Get parameters from the fitted model
        params = best_estimator.get_params()
        
        # 2. Determine the type and instantiate the correct wrapper
        if isinstance(best_estimator, SVC):
            return SVCModel(**params), 'svc'
        elif isinstance(best_estimator, SVR):
            return SVRModel(**params), 'svr'
        else:
            raise TypeError(f"Type unsupported for reconstruction: {type(best_estimator)}")

In [ ]:
if use_random:
    # --- 4.7.2 Reconstruction Logic ---

    final_models_list = []  # List to hold final model wrappers
    is_multi_output = False # Flag to indicate multi-output scenario

    # MLC25 case: check if final_model has 'estimators_' attribute typical of MultiOutput wrapper
    if hasattr(final_model, 'estimators_'):
        print(f"Detected Multi-Output System ({len(final_model.estimators_)} targets).")
        is_multi_output = True
        
        # Iterate over each RandomizedSearchCV contained in the MultiOutput
        for i, search_obj in enumerate(final_model.estimators_):
            # Extract the winner for this specific target
            best_est = search_obj.best_estimator_
            
            # Create wrapper
            wrapper, m_type = create_model_from_estimator(best_est)
            final_models_list.append(wrapper)
            
            print(f"Target {i}: Configured {m_type.upper()} with C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}, epsilon={getattr(wrapper.model, 'epsilon', 'N/A')}")

    # MONK case: Single Output
    else:
        print("Detected Single-Output System.")
        # final_model is directly the RandomizedSearchCV
        best_est = final_model.best_estimator_
        best_params = best_est.get_params()

        best_params['probability'] = True  
        
        wrapper, m_type = create_model_from_estimator(best_est)
        final_models_list.append(wrapper)
        
        print(f"Configured {m_type.upper()} with C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}")

In [ ]:
# === 5. OPTUNA HYPERPARAMETER OPTIMIZATION ===

# 5.1 Optuna Optimization Logic
if use_optuna:
    import optuna
    import warnings
    from sklearn.svm import SVR, SVC
    from sklearn.model_selection import cross_val_score
    from optuna.samplers import TPESampler
    

    # List to hold final model wrappers
    final_models_list = [] 
    best_cv_scores = [] 
    N_TRIALS = 200
    sampler = TPESampler(seed=seed)
    print(f"Starting Optuna Optimization ({N_TRIALS} trials)...")
    time_optuna_start = time.time()

    # --- ML-CUP ---
    if is_regression_task:
        print(">>> Optimizing Multi-Output SVR (One model per target)...")
        
        # Loop over each target for multi-output regression
        for i in range(y_train.shape[1]):
            print(f"\n--- Optimizing Target {i} ---")
            y_current = y_train[:, i]
            
            def objective_svr(trial):
                # Kernel choice
                kernel_choice = trial.suggest_categorical('kernel', ['rbf','poly','linear'])

                # Conditional parameters
                gamma_param = 'scale'  # Default for linear kernel
                degree_param = 3
                coef0_param = 0.0

                if kernel_choice == 'poly':
                    gamma_param = trial.suggest_float('gamma', 1e-3, GAMMA_LIMIT, log=True)
                    degree_param = trial.suggest_int('degree', 2, 5)
                    coef0_param = trial.suggest_float('coef0', 0.0, 10.0)
                elif kernel_choice == 'rbf':
                    gamma_param = trial.suggest_float('gamma', 1e-3, GAMMA_LIMIT, log=True)
                # 1. Search Space Definition
                param = {
                    'kernel': kernel_choice,
                    'C': trial.suggest_float('C', 1e-3, C_LIMIT, log=True),       
                    'epsilon': trial.suggest_float('epsilon', 1e-2, 1e1, log=True), 
                    'gamma': gamma_param,
                    'degree': degree_param,
                    'coef0': coef0_param
                }
                
                # 2. Model Instantiation
                model = SVR(**param, cache_size=1000, max_iter=100000)
                
                # 3. Validation  
                scores = cross_val_score(model, X_train, y_current, cv=kfold, scoring=mee_scorer, n_jobs=-1)
                return scores.mean() 

            # --- OPTUNA STUDY EXECUTION ---
            study = optuna.create_study(direction="maximize", sampler=sampler)
            study.optimize(objective_svr, n_trials=N_TRIALS, show_progress_bar=False)
            
            
            
            # --- SAVE CSV ---
            # Save trials to CSV
            try:
                csv_filename = f"svr_target_{i}_trials.csv"
                csv_path = os.path.join(run_dir, csv_filename)
                
                df_results = study.trials_dataframe()
                df_results.to_csv(csv_path, index=False)
                print(f"CSV saved: {csv_path}")
            except Exception as e:
                print(f"Could not generate plots for target {i}: {e}")

            best_cv_score = abs(study.best_value)
            best_cv_scores.append(best_cv_score)

            print(f"Best Params T{i}: {study.best_params}")
            
            # --- WRAPPER CREATION ---
            best_wrapper = SVRModel(**study.best_params, cache_size=1000, max_iter=100000)
            best_wrapper.fit(X_train, y_current) 
            
            final_models_list.append(best_wrapper)
            is_multi_output = True

    # --- MONK ---
    else:
        print(">>> Optimizing SVC (Single Output)...")
        
        def objective_svc(trial):
            kernel_choice = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])

            degree_param = 3
            coef0_param = 0.0
            gamma_param = 'scale'

            if kernel_choice == 'poly':
                degree_param = trial.suggest_int('degree', 2, 5)
                coef0_param = trial.suggest_float('coef0', 0.0, 10.0)
            elif kernel_choice != 'linear':
                gamma_param = trial.suggest_float('gamma', 1e-3, GAMMA_LIMIT, log=True)

            param = {
                'kernel': kernel_choice,
                'C': trial.suggest_float('C', 1e-3, C_LIMIT, log=True),
            }

            if kernel_choice != 'linear':
                param['gamma'] = gamma_param
            if kernel_choice == 'poly':
                param['degree'] = degree_param
                param['coef0'] = coef0_param
                
            model = SVC(**param, cache_size=1000, probability=True, random_state=seed)
            scores = cross_val_score(model, X_train, y_train.ravel(), cv=kfold, scoring='accuracy', n_jobs=-1)
            return scores.mean()

        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(objective_svc, n_trials=N_TRIALS, show_progress_bar=True)
        

        # 1. CSV
        try:
            csv_path = os.path.join(run_dir, "optuna_results_monk.csv")
            df_results = study.trials_dataframe()
            df_results.to_csv(csv_path, index=False)
            print(f"CSV saved: {csv_path}")
        except Exception as e:
            print(f"Could not generate plots: {e}")

        best_cv_scores.append(study.best_value)

        print(f"Best Params: {study.best_params}")
        
        # --- WRAPPER CREATION---
        best_wrapper = SVCModel(**study.best_params, cache_size=1000, probability=True, random_state=seed)
        best_wrapper.fit(X_train, y_train.ravel())
        
        final_models_list.append(best_wrapper)
        is_multi_output = False  # Flag for single-output
    end_time_optuna = time.time()
    time_optuna = end_time_optuna - time_optuna_start
    print(f"\nOptuna Optimization completed in {time_optuna:.2f}s")    
    print(f"\n Optimization Complete. Models ready in 'final_models_list': {len(final_models_list)}")

    

In [ ]:
# 5.2 Print Final Models Summary

if use_optuna:
    # Print final models summary
    if is_multi_output:
        for i, model in enumerate(final_models_list):
            print(f"Target {i}: Model with params: {model.model.get_params()}")
    else:
        print(f"Single Target Model with params: {final_models_list[0].model.get_params()}")

In [ ]:
# === 6. FINAL FITTING ===

print("\n--- Starting Final Refitting of Models ---")

if is_multi_output:
    for i, wrapper in enumerate(final_models_list):
        print(f"Fitting Target {i}...")
        # Note: y_train[:, i] takes only the i-th column
        wrapper.fit(X_train, y_train[:, i]) 
        
else:
    print("Fitting Single Model...")
    final_models_list[0].fit(X_train, y_train.ravel())

print("Refitting done.")

In [ ]:
# === 7. PREDICTION & EVALUATION ===
from sklearn.metrics import mean_squared_error, accuracy_score

# 7.1 & 7.2 Generation of Predictions
print("\n--- Generating Predictions ---")

if is_multi_output:
    # Multi-Output Case (Regression ML-CUP)
    train_preds = [m.predict(X_train) for m in final_models_list]
    y_train_pred = np.column_stack(train_preds)
    
    test_preds = [m.predict(X_test) for m in final_models_list]
    y_test_pred = np.column_stack(test_preds)
    
    y_train_eval = y_train
    y_test_eval = y_test
else:
    # Single Output Case (Classification MONK)
    y_train_pred = final_models_list[0].predict(X_train)
    y_test_pred = final_models_list[0].predict(X_test)
    
    y_train_eval = y_train.ravel()
    y_test_eval = y_test.ravel()

# 7.3 Calculation of Final Metrics
print("\n--- 7.3 Calculating Final Metrics ---")

final_train_accuracy = None
final_test_loss = None

if is_regression_task:
    # --- REGRESSION (MEE) ---
    final_train_metric = mean_euclidean_error(y_train_eval, y_train_pred)
    final_test_metric = mean_euclidean_error(y_test_eval, y_test_pred)
    
    # Mapping variables for saving
    final_train_loss = final_train_metric
    final_test_score = final_test_metric
    metric_label = "MEE"
    
    print(f"Final Training MEE: {final_train_metric:.5f}")
    print(f"Final Test MEE:     {final_test_metric:.5f}")

else:
    # --- CLASSIFICATION (MSE + ACCURACY) ---
    
    # 1. MSE Calculation
    train_mse = mean_squared_error(y_train_eval, y_train_pred)
    test_mse = mean_squared_error(y_test_eval, y_test_pred)
    
    # 2. Accuracy Calculation
    train_acc = accuracy_score(y_train_eval, y_train_pred) * 100.0
    test_acc = accuracy_score(y_test_eval, y_test_pred) * 100.0
    
    # Mapping variables for saving
    final_train_loss = train_mse   
    final_test_score = test_acc    
    final_train_accuracy = train_acc
    final_test_loss = test_mse
    metric_label = "Accuracy"

    print(f"Training MSE: {train_mse:.5f} | Accuracy: {train_acc:.2f}%")
    print(f"Test MSE:     {test_mse:.5f} | Accuracy: {test_acc:.2f}%")

In [ ]:
# 7.4 Baseline Comparison (Dummy Regressor for Regression Tasks)
from sklearn.dummy import DummyRegressor
from sklearn.metrics import make_scorer

if is_regression_task:
    # 1. Define dummy model
    # strategy='mean' predicts always the mean of the training set for each target
    dummy_regr = DummyRegressor(strategy="mean")

    # 2. Training (just computes means, it's instantaneous)
    print("\n--- Training Baseline (Dummy Regressor) ---")
    dummy_regr.fit(X_train, y_train)

    # 3. Prediction
    y_pred_dummy = dummy_regr.predict(X_test)

    # 4. Baseline MEE Calculation
    mee_baseline = mean_euclidean_error(y_test, y_pred_dummy)

    print(f"Baseline MEE (Mean Strategy): {mee_baseline:.4f}")
    print(f"Best SVR model MEE:          {final_test_metric:.4f}") 

    # 5. Comparison
    if final_test_metric < mee_baseline:
        print("SUCCESS: the model outperforms the baseline!")
        improvement = mee_baseline - final_test_metric
        print(f"   Improvement: {improvement:.4f} MEE points")
    else:
        print("FAIL: the model is worse than the baseline.")

In [ ]:
# 7.5 Baseline Comparison (Dummy Classifier for Classification Tasks)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

if not is_regression_task:
    print("\n--- Training Baseline (Dummy Classifier) ---")
    
    # 1. Define dummy model
    # strategy='most_frequent' always predicts the most frequent class in the training set
    dummy_clf = DummyClassifier(strategy="most_frequent")
    
    # 2. Training 
    dummy_clf.fit(X_train, y_train.ravel())
    
    # 3. Prediction
    y_pred_dummy = dummy_clf.predict(X_test)
    
    # 4. Baseline Accuracy Calculation
    acc_baseline = accuracy_score(y_test.ravel(), y_pred_dummy) * 100.0
    
    print(f"Baseline Accuracy (Most Frequent): {acc_baseline:.2f}%")
    print(f"Best SVC model Accuracy:          {final_test_score:.2f}%") 
    
    # 5. Comparison
    if final_test_score > acc_baseline:
        print("SUCCESS: the model outperforms the baseline!")
        print(f"   Improvement: +{final_test_score - acc_baseline:.2f}%")
    else:
        print("FAIL: the model is worse than the baseline.")

In [ ]:
# 7.6 LEARNING CURVE VISUALIZATION (UNIVERSAL)
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np
import os

# Custom Multi-Target Wrapper for Global Curve (Regression)
class MultiTargetSVRSystem(BaseEstimator, RegressorMixin):
    def __init__(self, estimators_list):
        self.estimators_list = estimators_list
        self.fitted_estimators_ = []
    def fit(self, X, y):
        self.fitted_estimators_ = []
        y_np = y.values if hasattr(y, 'values') else y
        for i, wrapper in enumerate(self.estimators_list):
            model_clone = clone(wrapper.model)
            model_clone.fit(X, y_np[:, i])
            self.fitted_estimators_.append(model_clone)
        return self
    def predict(self, X):
        predictions = [m.predict(X) for m in self.fitted_estimators_]
        return np.column_stack(predictions)

print("\n--- Generating Learning Curve ---")

try:
    if 'final_models_list' in globals() and final_models_list:

        if is_regression_task:
            estimator = MultiTargetSVRSystem(final_models_list)
            scoring_metric = mee_scorer 
            ylabel = "Mean Euclidean Error (MEE)"
            title = f"Global Learning Curve (System MEE) - {dataset_name.upper()}"
            
            # Benchmark retrieval
            base_score = mee_baseline if 'mee_baseline' in globals() else None
            test_score = final_test_metric if 'final_test_metric' in globals() else None
            
            invert_sign = True 
            
        else:
            estimator = final_models_list[0].model 
            scoring_metric = 'accuracy'
            ylabel = "Accuracy (%)"
            title = f"Classification Learning Curve (Accuracy) - {dataset_name.upper()}"
            
            # Benchmark retrieval
            base_val = acc_baseline if 'acc_baseline' in globals() else 0
            test_val = final_test_score if 'final_test_score' in globals() else 0
            
            # Normalize to percentage if needed
            base_score = base_val if base_val > 1 else base_val * 100
            test_score = test_val if test_val > 1 else test_val * 100
            invert_sign = False

        # --- LEARNING CURVE CALCULATION ---
        train_sizes, train_scores, test_scores = learning_curve(
            estimator, 
            X_train, y_train if is_regression_task else y_train.ravel(), 
            cv=kfold, 
            scoring=scoring_metric, 
            n_jobs=-1, 
            train_sizes=np.linspace(0.1, 1.0, 5)
        )

        # --- PLOTTING ---
        train_mean = np.mean(train_scores, axis=1)
        test_mean = np.mean(test_scores, axis=1)
        
        if invert_sign: # MEE
            if train_mean[0] < 0:
                train_mean = -train_mean
                test_mean = -test_mean
        else: # Accuracy
            train_mean = train_mean * 100
            test_mean = test_mean * 100

        train_std = np.std(train_scores, axis=1)
        test_std = np.std(test_scores, axis=1)
        if not invert_sign:
            train_std *= 100
            test_std *= 100

        plt.figure(figsize=(10, 7))
        plt.title(title)
        plt.xlabel("Training Examples")
        plt.ylabel(ylabel)
        
        plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training Score")
        plt.plot(train_sizes, test_mean, 'o-', color="g", label="Validation Score (CV)")
        
        plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
        plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
        
        if base_score is not None:
            plt.axhline(y=base_score, color='grey', linestyle='--', label=f"Baseline: {base_score:.2f}")
        if test_score is not None:
            plt.axhline(y=test_score, color='blue', linestyle='-.', label=f"Test Set: {test_score:.2f}")
        
        plt.grid(True)
        plt.legend(loc="best")
        
        if 'run_dir' in globals():
            plt.savefig(os.path.join(run_dir, "Learning_Curve.png"))
            print(f"   Saved Plot: {os.path.join(run_dir, 'Learning_Curve.png')}")
        
        plt.show()

    else:
        print("Final models list not found. Skipping Learning Curve.")

except Exception as e:
    print(f"Could not generate Learning Curve: {e}")

In [ ]:
# === 8. SYSTEM SAVING ===
import os
import json
import joblib

# 8.1 Saving Function Definition
def save_model_system(models_list, scaler, pca_obj, dataset_name, is_regression, optimization_method, 
                      final_train_score, final_test_score, cv_scores_list, seed,
                      target_dir, final_train_accuracy = None, final_test_loss = None, search_time=None):  
    """
    Saves the trained models, scaler, and updates the registry JSON file.
    Parameters:
    - models_list: List of trained model wrappers (SVCModel or SVRModel).
    - scaler: The fitted scaler object (e.g., StandardScaler).
    - dataset_name: Name of the dataset (e.g., 'mlc25', 'monk1', 'monk2', 'monk3').
    - is_regression: Boolean indicating if the task is regression.
    - optimization_method: String indicating the optimization method used ('Optuna' or 'RandomSearch').
    - final_train_score: Final training score (float).
    - final_test_score: Final test score (float).
    - cv_scores_list: List of cross-validation scores for each target/model.
    - seed: Random seed used for reproducibility.
    - target_dir: Directory where to save the models and registry.
    ️- final_train_accuracy: Final training accuracy (for classification tasks).
    - final_train_loss: Final training loss (for regression tasks).
    - search_time: Time taken for hyperparameter search (in seconds).
    """
    # Extract run name from target_dir
    run_name = os.path.basename(target_dir)
    
    # Define registry file path
    method_folder = os.path.dirname(target_dir)
    registry_file = os.path.join(method_folder, f"registry.json")

    # 1. Load Registry
    registry = {}
    if os.path.exists(registry_file):
        try:
            with open(registry_file, 'r') as f:
                registry = json.load(f)
        except json.JSONDecodeError:
            print("Warning: Registry corrupted. Starting fresh.")
    
    # 2. Save Scaler 
    scaler_path = os.path.join(target_dir, "scaler.pkl")
    joblib.dump(scaler, scaler_path)
    print(f"Scaler saved to: {scaler_path}")

    # Save PCA if exists
    if pca_obj is not None:
        pca_path = os.path.join(target_dir, "pca_model.pkl")
        joblib.dump(pca_obj, pca_path)
        print(f"PCA model saved to: {pca_path}")

    # 3. Construct JSON Entry
    json_entry = {
        "type": "MultiOutput SVR" if is_regression else "SVC",
        "method": optimization_method,
        "seed": seed,
        "location": target_dir,
        "pca_active": pca_obj is not None,
        "train_loss": round(final_train_score, 4),
        "train_accuracy": round(final_train_accuracy, 4) if final_train_accuracy is not None else None,
        "test_loss": round(final_test_loss, 4) if final_test_loss is not None else None,
        "global_test_score": round(final_test_score, 4),
        "metric_name": "MEE" if is_regression else "Accuracy",
        "search_time_seconds": search_time,
        "targets": {} 
    }

    # 4. Save Models
    if is_regression:
        for i, wrapper in enumerate(models_list):
            filename = f"target_{i}.pkl"
            wrapper.save_pkl(os.path.join(target_dir, filename))
            
            train_t = getattr(wrapper, 'train_time', None)

            json_entry["targets"][f"target_{i}"] = {
                "cv_score": round(cv_scores_list[i], 4),
                "train_time_seconds": round(train_t, 4) if train_t is not None else None,
                "params": wrapper.get_params_clean()
            }
    else:
        wrapper = models_list[0]
        wrapper.save_pkl(os.path.join(target_dir, "classifier_model.pkl"))
        
        train_t = getattr(wrapper, 'train_time', None)
        
        json_entry["targets"]["classifier"] = {
            "cv_score": round(cv_scores_list[0], 4),
            "train_time_seconds": round(train_t, 4) if train_t is not None else None,
            "params": wrapper.get_params_clean()
        }

    # 5. Write Registry
    registry[run_name] = json_entry
    with open(registry_file, 'w') as f:
        json.dump(registry, f, indent=4)
        
    print(f"Registry updated: {registry_file}")
    print(f"System Saved in: {target_dir}")



In [ ]:
# 8.2 Saving Execution

current_method = "Optuna" if use_optuna else "RandomSearch" if use_random else "GridSearch"
time = time_grid if use_grid else time_random if use_random else time_optuna

save_model_system(
    models_list=final_models_list, 
    scaler=scaler,
    pca_obj=pca_model,
    dataset_name=dataset_name, 
    is_regression=is_regression_task,
    optimization_method=current_method,
    final_train_score=final_train_loss,
    final_test_score=final_test_score,
    cv_scores_list=best_cv_scores,
    seed=seed,
    target_dir=run_dir,
    final_train_accuracy=final_train_accuracy,
    final_test_loss=final_test_loss,
    search_time=time   
)


In [ ]:
# 9. VISUALIZATIONS (INFERENCE ANALYSIS)
from sklearn.metrics import PredictionErrorDisplay
import matplotlib.pyplot as plt
import numpy as np
import os

print("\n--- Generating Inference Analysis Plots ---")

if 'y_test_pred' in globals() and 'final_models_list' in globals():
    
    # MLC25: Regression Analysis
    if is_regression_task:
        print(">> Type: Regression Analysis (Residuals & Scatter)")
        
        y_true_np = y_test.values if hasattr(y_test, 'values') else y_test
        y_pred_np = y_test_pred
        scatter_kws = {"alpha": 0.3, "color": "tab:blue"}
        line_kws = {"color": "tab:red", "linestyle": "--"}

        # Plot Global
        y_true_flat = y_true_np.flatten()
        y_pred_flat = y_pred_np.flatten()
        
        fig, axs = plt.subplots(ncols=2, figsize=(14, 6))
        PredictionErrorDisplay.from_predictions(y_true=y_true_flat, y_pred=y_pred_flat, kind="actual_vs_predicted", ax=axs[0], scatter_kwargs=scatter_kws, line_kwargs=line_kws)
        axs[0].set_title(f"Global: Actual vs Predicted")
        PredictionErrorDisplay.from_predictions(y_true=y_true_flat, y_pred=y_pred_flat, kind="residual_vs_predicted", ax=axs[1], scatter_kwargs=scatter_kws, line_kwargs=line_kws)
        axs[1].set_title(f"Global: Residuals Analysis")
        plt.tight_layout()
        if 'run_dir' in globals(): plt.savefig(os.path.join(run_dir, "Scatter_Global.png"))
        plt.show()

        # Plot Per Target
        for i in range(y_true_np.shape[1]):
            # Target-specific plots
            y_t = y_true_np[:, i]
            y_p = y_pred_np[:, i]
            fig, axs = plt.subplots(ncols=2, figsize=(14, 6))
            PredictionErrorDisplay.from_predictions(y_true=y_t, y_pred=y_p, kind="actual_vs_predicted", ax=axs[0], scatter_kwargs=scatter_kws, line_kwargs=line_kws)
            axs[0].set_title(f"Target {i}: Actual vs Predicted")
            PredictionErrorDisplay.from_predictions(y_true=y_t, y_pred=y_p, kind="residual_vs_predicted", ax=axs[1], scatter_kwargs=scatter_kws, line_kwargs=line_kws)
            axs[1].set_title(f"Target {i}: Residuals")
            plt.tight_layout()
            if 'run_dir' in globals(): plt.savefig(os.path.join(run_dir, f"Scatter_Target_{i}.png"))
            plt.show()
            plt.close()
            
    # MONK: Classification Analysis
    else:
        print(">> Type: Classification Analysis (Using SVCModel internal method)")
        
        # Take the classifier wrapper
        clf_wrapper = final_models_list[0]
        
        # Use the built-in method to plot classification analysis
        clf_wrapper.plot_classification_analysis(X_test, y_test.ravel())
        
        # Note: The method handles saving internally if run_dir is set
        print("(Plot displayed using 'plot_classification_analysis' from SVCModel)")

else:
    print("Skipping Visualizations (Data or Models missing).")

In [ ]:
# === 10. OPTUNA VISUALIZATION (FROM CSV) ===

if use_optuna:
    print("\n--- Generating Optuna Plots from Saved CSVs ---")
    
    # 1. Helper Function to Import Study and Plot
    from utils.optuna_utils import import_csv
    from optuna.visualization import plot_optimization_history, plot_param_importances
    
    def show_optuna_plots(study, title_prefix):
        print(f"\n Visualizing: {title_prefix}")
        
        # History
        fig_hist = plot_optimization_history(study)
        fig_hist.update_layout(title=f"{title_prefix} - Optimization History")
        fig_hist.show()
        
        # Importance
        try:
            fig_imp = plot_param_importances(study)
            fig_imp.update_layout(title=f"{title_prefix} - Hyperparameter Importance")
            fig_imp.show()
        except Exception as e:
            print(f"Importance plot skipped: {e}")

    # --- EXECUTION ---
    if is_regression_task:
        # Loop over each target for ML-CUP
        for i in range(y_train.shape[1]):
            csv_name = f"svr_target_{i}_trials.csv"
            csv_path = os.path.join(run_dir, csv_name)
            
            if os.path.exists(csv_path):
                # Import study from CSV with maximize due to negative MEE
                study = import_csv(csv_path, direction="maximize")
                show_optuna_plots(study, title_prefix=f"Target {i}")
            else:
                print(f"CSV not found: {csv_path}")

    else:
        # Monk case
        csv_name = "optuna_results_monk.csv"
        csv_path = os.path.join(run_dir, csv_name)
        
        if os.path.exists(csv_path):
            study = import_csv(csv_path, direction="maximize")
            show_optuna_plots(study, title_prefix=f"{dataset.upper()} Classification")
        else:
            print(f"CSV not found: {csv_path}")

# --- EXTRA. SAVE FINAL MODELS FOR ENSEMBLE ---
import joblib
import os

os.makedirs("models", exist_ok=True)

if 'final_models_list' in globals():
    if is_multi_output:
        # Save the list of wrappers
        save_path = "models/best_svm_mlcup_list.joblib"
        joblib.dump(final_models_list, save_path)
        print(f"Multi-output SVM models saved to {save_path}")
    else:
        # Save the single wrapper
        save_path = "models/best_svm_mlcup_single.joblib"
        joblib.dump(final_models_list[0], save_path)
        print(f"Single SVM model saved to {save_path}")
else:
    print("Warning: final_models_list not found, cannot save models.")


In [ ]:
# --- EXTRA. LOG RESULTS TO ALL_RESULTS.JSON ---
# This section updates a global results JSON file with final metrics. Useful in "compare_models.ipynb"
from sklearn.metrics import accuracy_score
import numpy as np

# Ensure metrics are calculated
if 'is_regression_task' in globals() and is_regression_task:
    # CUP - MEE
    # Check availability of y_test_pred and y_test_eval from 'Prediction & Evaluation' section
    if 'y_test_pred' in globals() and 'y_test_eval' in globals():
         diff = y_test_eval - y_test_pred
         # simple euclidean norm
         mee = np.mean(np.linalg.norm(diff, axis=1))
         print(f"Logging MEE to all_results.json: {mee}")
         update_json("CUP", "SVM", {"internal_test_MEE": mee})
    else:
         print("Warning: y_test_pred or y_test_eval not found. Cannot log MEE.")
else:
    # Monk - Accuracy
    if 'y_test_pred' in globals() and 'y_test_eval' in globals():
        acc = accuracy_score(y_test_eval, y_test_pred)
        
        # Determine Task Name correctly
        task_map = {'monk1': 'Monk-1', 'monk2': 'Monk-2', 'monk3': 'Monk-3'}
        ds_name = dataset if 'dataset' in globals() else 'unknown'
        task_name = task_map.get(ds_name, ds_name)
        
        if ds_name == 'monk3' and 'monk3_reg' in globals() and monk3_reg:
             task_name = 'Monk-3-Reg'
        
        print(f"Logging Accuracy to all_results.json for {task_name}: {acc}")
        update_json(task_name, "SVM", {"test_accuracy": acc})
    else:
         print("Warning: y_test_pred or y_test_eval not found. Cannot log Accuracy.")
